# 02 · Campaign — cofactor site → scaffold → cofactor-aware LigandMPNN (coordinating residues fixed)

**Standard slot:** *design campaign.* **For Project 24 this means:** take the cofactor-site spec,
scaffold it into many backbones (RFdiffusion2 / Riff-Diff — the **A100** step), then **cofactor-aware
LigandMPNN sequence design fixing the coordinating residues** (and passing the cofactor as atom
context), and write a results CSV (D2).

Runs end-to-end on the **mock** backend with no GPU; switch to the real backends on Colab/HPC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams
Tools change. Before a campaign, HTTP-check that the pinned upstream repos still exist, and pin the
commit/tag you actually use. **RFdiffusion2 and Riff-Diff are new and move fast — VERIFY the current
public release/repo at generation time** (do not assert a repo you are unsure of); the others below
are stable enough to head-check.

In [ ]:
import requests

# Pinned upstreams (pin the COMMIT/TAG you use in env/requirements.txt + LOG.md):
STABLE_UPSTREAMS = {
    "LigandMPNN (cofactor-aware, coordinating-residue-fixed seq design)": "https://github.com/dauparas/LigandMPNN",
    "RFdiffusion (classic motif scaffolding — free-tier demo)": "https://github.com/RosettaCommons/RFdiffusion",
    "ColabFold (AF2 — site pLDDT + apo backbone)": "https://github.com/sokrypton/ColabFold",
    "AutoDock Vina (cofactor fit)": "https://github.com/ccsb-scripps/AutoDock-Vina",
}
# VERIFY-ONLY (new/fast-moving; confirm the current release before relying on a URL):
VERIFY_UPSTREAMS = [
    "RFdiffusion2 (Dauparas 2025) — VERIFY current public release/repo at generation time",
    "Riff-Diff (Schnettler 2025, Nature) — VERIFY current public release/repo at generation time",
]

for name, url in STABLE_UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"[{r.status_code}] {name}\n      {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e!r}\n      {url}")
print("\nVERIFY MANUALLY (do not assert a repo URL you are unsure of):")
for v in VERIFY_UPSTREAMS:
    print("  -", v)

## 1 · Build the cofactor site and scaffold its coordination pocket
The mock path returns placeholder backbones so the loop runs anywhere. On an A100, switch `METHOD` to
`"rfdiffusion2"` or `"riffdiff"` (verify the release) and `N_SCAFFOLDS` to 1000s.

> **A100 NOTE:** scaffolding 1000s of backbones is the compute bottleneck. Free Colab T4 can do a
> small **RFdiffusion** (classic) motif-scaffolding demo (tens of backbones); the real campaign wants
> an A100 (Colab Pro+) or HPC. A **cofactor pocket is harder than a single-sidechain motif** — e.g.
> two axial His on *opposite* helices at the right Fe distance/angle — so budget extra backbones
> because many will not hold a clean coordination geometry. The mock backend below needs no GPU.

In [ ]:
from cofactor_tools import build_cofactor_spec, scaffold_cofactor_pocket

COFACTOR = "heme"          # heme | fes | zn  (alias -> a coordination scheme)
SCHEME = "bis_his_heme"    # the project default; see cofactor_tools.COORDINATION_SCHEMES for others
spec = build_cofactor_spec(COFACTOR, scheme=SCHEME)

METHOD = "mock"            # -> "rfdiffusion2" | "riffdiff" | "rfdiffusion" on Colab/HPC (verify release)
N_SCAFFOLDS = 12           # -> 1000s for the real campaign

scaffolds = scaffold_cofactor_pocket(spec, n=N_SCAFFOLDS, tool=METHOD)
print(f"{len(scaffolds)} scaffolds via tool={METHOD!r}, scheme={SCHEME!r} (mock numbers are SYNTHETIC)")
print("example:", scaffolds[0])

## 2 · Cofactor-aware LigandMPNN — FIXING the coordinating residues
This is the **central tool** of the project and why **LigandMPNN, not vanilla ProteinMPNN**, is used:
the coordinating residues (e.g. the two axial His of a bis-His heme) must hold the cofactor, so (a)
those positions are **FIXED** and (b) the cofactor (heme / cluster / metal) is **passed as atom
context** so the model designs the rest of the protein to accommodate — and not clash with — the
bound cofactor. ProteinMPNN cannot see the cofactor. On Colab set `TOOL="ligandmpnn"` (CPU-fast),
pass `--ligand_mpnn_use_atom_context 1`, and a fixed-positions list covering **every** coordinating
residue.

In [ ]:
from cofactor_tools import ligandmpnn_cofactor

TOOL = "mock"              # -> "ligandmpnn" on Colab (CPU-fast)
SEQS_PER_BACKBONE = 4

coordinating = spec.coordinating_residue_ids()
all_designs = []
for bb in scaffolds:
    seqs = ligandmpnn_cofactor(bb, coordinating, n=SEQS_PER_BACKBONE,
                               cofactor=spec.cofactor, tool=TOOL)
    for s in seqs:
        s["scaffold_id"] = bb["design_id"]
        s["scaffold_tool"] = bb["tool"]
        s["motif_rmsd"] = bb["motif_rmsd"]
        all_designs.append(s)
print(f"{len(all_designs)} sequences total "
      f"({len(scaffolds)} backbones x {SEQS_PER_BACKBONE}); coordinating roles fixed: {coordinating}")
print("sanity: every sequence carries the fixed coordinating positions ->",
      all_designs[0]["fixed_positions"])

## 3 · Predict + score (mock coordination-geometry RMSD), write the results CSV
On Colab, predict each sequence with AF2/ESMFold, read the **site pLDDT** (at the coordinating
residues), **place the metal/cofactor by docking/superposition** (AF2 gives the apo backbone), and
compute the real `coordination_geometry` from the predicted PDB. Here the mock backend fills SYNTHETIC
values so the CSV — the input to notebook 03 — is produced anywhere.

In [ ]:
import pandas as pd
from cofactor_tools import coordination_geometry, dock_cofactor, cofactor_site_md

rows = []
for d in all_designs:
    # On Colab, `pred_pdb` is the AF2-predicted (apo) PDB path for this design; the mock backend keys
    # off the (non-existent) per-design path string so each design gets a DISTINCT SYNTHETIC value.
    pred_pdb = f"results/pred/{d['design_id']}.pdb"
    cg = coordination_geometry(pred_pdb, spec)         # mock -> SYNTHETIC (varies per design)
    dock = dock_cofactor(pred_pdb, spec.cofactor)      # mock -> SYNTHETIC (fit/orientation only)
    md_res = cofactor_site_md(pred_pdb, ns=10.0)       # mock -> SYNTHETIC (CAVEATED metal-FF proxy)
    # SYNTHETIC stand-ins for AF2 confidence so the plumbing runs (replace with real predictions):
    import hashlib
    h = int(hashlib.sha256(d["design_id"].encode()).hexdigest(), 16)
    plddt = 78 + (h % 20)            # 78-97, SYNTHETIC (global)
    plddt_site = 80 + ((h >> 7) % 18) # 80-97, SYNTHETIC (at the coordinating residues)
    scrmsd = round(0.8 + ((h >> 11) % 200) / 100.0, 2)  # 0.8-2.8, SYNTHETIC
    rows.append(dict(
        design_id=d["design_id"], scaffold_id=d["scaffold_id"],
        scaffold_tool=d["scaffold_tool"], cofactor=d["cofactor"], sequence=d["sequence"],
        plddt=plddt, plddt_site=plddt_site, scrmsd=scrmsd,
        coordination_geom_rmsd=cg, vina_score=dock["vina_score"],
        fe_between_ligands=dock["fe_between_ligands"], md_rmsd=md_res["md_rmsd"], synthetic=True))

camp = pd.DataFrame(rows)
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape, "(ALL NUMBERS SYNTHETIC — mock backend)")
print(f"coordination_geom_rmsd range: "
      f"{camp['coordination_geom_rmsd'].min()}-{camp['coordination_geom_rmsd'].max()} A")
camp.head()

## D2 checklist
- [ ] Scaffolding run logged (tool, release/commit, N backbones, seed) — A100 for the real campaign.
- [ ] Cofactor-aware LigandMPNN sequences with the **coordinating residues provably fixed**
      (fixed-positions list logged) **and the cofactor passed as atom context**.
- [ ] `results/campaign.csv` with one row per design (real metrics on Colab; mock here).
- [ ] Design log (every config + seed + output path) + 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared enzyme filter on `campaign.csv`.